# GPU benchmark on the verified NZ snapshot

This notebook does exactly one thing: **measure** training time/epoch, validation time, and peak GPU memory for a short, properly batched run on the real, verified snapshot -- not estimate, measure. Nothing here commits to a full training run; that decision comes after these numbers exist, once the data has been confirmed to load correctly here too.

Prerequisite (done on the local machine, not here): `data/nz_pilot_snapshot` was built by `scripts/colab/build_snapshot.py` -- every tile independently checksum- and schema-verified, cross-split geographic separation and parcel-id overlap both confirmed clean. Zip it and upload to Drive before running this:

```bash
cd /path/to/SIH_2026
zip -r nz_pilot_snapshot.zip data/nz_pilot_snapshot
```
Then upload `nz_pilot_snapshot.zip` to `My Drive/` (the Drive app or web uploader is far faster than uploading through the Colab file browser for a multi-GB file).

In [ ]:
!git clone https://github.com/SatvikSaluja/SIH_2026.git
%cd SIH_2026
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

In [ ]:
# Verify CUDA is ACTUALLY available before anything else -- a benchmark
# run silently on CPU because the runtime type wasn't set to GPU would
# produce numbers that look like a GPU result but aren't.
import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > select a GPU, then rerun"
print("GPU:", torch.cuda.get_device_name(0))
print("Total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/nz_pilot_snapshot.zip -d /content/
!ls /content/nz_pilot_snapshot | wc -l  # should be ~710 (708 train + val + test + manifest.json)

In [ ]:
# Confirm the data actually loads correctly HERE before trusting any
# timing number from it -- a manifest/schema mismatch that somehow
# didn't surface locally must not surface for the first time inside a
# timed run.
import sys
sys.path.insert(0, '/content/SIH_2026/scripts/colab')
from train_real import Patches
train = Patches('/content/nz_pilot_snapshot', 'train')
val = Patches('/content/nz_pilot_snapshot', 'val')
print(f"{len(train)} training patches, {len(val)} validation patches -- loaded without error")

In [ ]:
# The actual measurement. batch_size=32 is a starting point, not tuned --
# the point of this cell is the numbers it prints, not this choice.
!python /content/SIH_2026/scripts/colab/benchmark_gpu.py \
  --data /content/nz_pilot_snapshot \
  --epochs 3 \
  --batch-size 32

## Reading the result

`benchmark_result.json` (also printed above) has `mean_train_seconds_per_epoch`, `mean_val_seconds`, and `peak_gpu_memory_mb`, measured, not guessed. To project a full run's wall-clock time honestly:

```
estimated_full_run = (target_epochs / 3) * mean_train_seconds_per_epoch
```

using the SAME 708-tile snapshot -- this notebook does not extrapolate to a larger tile count, because that would be a second, different assumption stacked on top of a measured one. If more tiles are added later, rerun this notebook against the new snapshot rather than scaling this number.

**Do not treat a clean run here as license to start a long unattended job.** Check `peak_gpu_memory_mb` against the GPU's total memory (cell 2) with headroom for a larger batch size if one gets used later, and look at whether `val_loss` moved sensibly across the 3 epochs -- a NaN or a value that exploded means something is still wrong with the data or setup, not something a longer run will fix.